<a href="https://colab.research.google.com/github/tele-boy/solution_for_p1428/blob/main/CS461_RO(EX1)_Lab1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# CS461 Robotics Lab 1: Imitation Learning

In this lab, you will implement:
1. **Behavior Cloning (BC)** with a Gaussian policy
2. **DAgger** (Dataset Aggregation)
3. **Run experiments** (Reacher & PointMaze) and report the success rate.
4. **Optional: Other imitation learning method** (extra credit)

You will train and evaluate on two environments: **Reacher** and **PointMaze**.

**Submission**: **A completed jupyter notebook** (you may also append **a lab report document** for detailed results, using a A4-page document in docx or pdf format) due on or before **May 12 23:59 UTC+8**

In [1]:
# @title User Input Form
# @markdown Enter your name below:

from IPython.display import display, HTML

name = "Jing Ning, Xie" # @param {type:"string"}
number = "1230018232" # @param {type:"string"}

print(f"Student Name: {name}")
print(f"Student Number: {number}")


if name:
    display(HTML(f'<h1 style="color: green;"><b>Student Name: ✨ {name} ✨</b></h1>'))
else:
    display(HTML('<p style="color: red;">Please enter your name above</p>'))

if number:
    display(HTML(f'<h1 style="color: green;"><b>Student No. :  {number} </b></h1>'))
else:
    display(HTML('<p style="color: red;">Please enter your student number above</p>'))

Student Name: Jing Ning, Xie
Student Number: 1230018232


## 1. Setup and Installation - upload the lab1.zip and extract it.

In [2]:
from google.colab import files
uploaded = files.upload()

Saving lab1.zip to lab1.zip


In [3]:
!unzip lab1.zip
!cp -r lab1/* .

Archive:  lab1.zip
   creating: lab1/
  inflating: __MACOSX/._lab1         
  inflating: lab1/pytorch_utils.py   
  inflating: __MACOSX/lab1/._pytorch_utils.py  
   creating: lab1/reach_goal/
  inflating: __MACOSX/lab1/._reach_goal  
  inflating: lab1/policy.py          
  inflating: __MACOSX/lab1/._policy.py  
  inflating: lab1/utils.py           
  inflating: __MACOSX/lab1/._utils.py  
  inflating: lab1/evaluate.py        
  inflating: __MACOSX/lab1/._evaluate.py  
   creating: lab1/data/
  inflating: __MACOSX/lab1/._data    
   creating: lab1/reach_goal/resources/
  inflating: __MACOSX/lab1/reach_goal/._resources  
  inflating: lab1/reach_goal/__init__.py  
  inflating: __MACOSX/lab1/reach_goal/.___init__.py  
   creating: lab1/reach_goal/envs/
  inflating: __MACOSX/lab1/reach_goal/._envs  
  inflating: lab1/data/pointmaze_expert_data.pkl  
  inflating: __MACOSX/lab1/data/._pointmaze_expert_data.pkl  
  inflating: lab1/data/reacher_expert_policy.pkl  
  inflating: __MACOSX/lab1/data

In [ ]:
# Install system dependencies (for MuJoCo rendering)
!apt-get install -y \
    libgl1-mesa-dev \
    libgl1-mesa-glx \
    libglew-dev \
    libosmesa6-dev \
    software-properties-common \
    patchelf

# Install PyTorch with CUDA support (separate to avoid index conflicts)
!pip install -q \
    torch==2.6.0 torchvision==0.21.0 torchaudio==2.6.0 \
    --extra-index-url https://download.pytorch.org/whl/cu124

# Install mujoco first — latest has Python 3.12 prebuilt wheels.
# gymnasium-robotics 1.2.x pins mujoco<3.0, but those old versions lack
# cp312 wheels. We install mujoco separately, then gymnasium-robotics with
# --no-deps to avoid the downgrade, and manually install its other deps.
!pip install -q mujoco
!pip install -q "gymnasium-robotics>=1.2.0,<1.3.0" --no-deps
!pip install -q \
    gymnasium==0.29.1 \
    Jinja2 imageio PettingZoo \
    matplotlib \
    "numpy>=1.24,<2.0" \
    tqdm

import os
os.environ['LD_PRELOAD'] = ':/usr/lib/x86_64-linux-gnu/libGLEW.so'

# Restart the runtime so the new numpy version takes effect.
# After restart, skip this cell and continue from the next one.
os.kill(os.getpid(), 9)

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
libglew-dev is already the newest version (2.2.0-4).
patchelf is already the newest version (0.14.3-1).
libgl1-mesa-dev is already the newest version (23.2.1-1ubuntu3.1~22.04.3).
libosmesa6-dev is already the newest version (23.2.1-1ubuntu3.1~22.04.3).
software-properties-common is already the newest version (0.99.22.9).
libgl1-mesa-glx is already the newest version (23.0.4-0ubuntu1~22.04.1).
0 upgraded, 0 newly installed, 0 to remove and 42 not upgraded.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 1.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 6.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 13.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 6.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

**Important:** The cell above restarts the runtime to load the new numpy version. After it restarts, **do not re-run the install cell** — skip it and continue from the next cell.

** Restart the runtime

In [1]:
import numpy as np
print(np.__version__)

1.26.4


In [2]:
# Re-set env var (lost after runtime restart)
import os
os.environ['LD_PRELOAD'] = ':/usr/lib/x86_64-linux-gnu/libGLEW.so'

# Verify installation — expect output: (11,)
import gymnasium as gym
env = gym.make("Reacher-v4")
print(env.reset()[0].shape)
env.close()

(11,)


## 2. Imports and Device Setup

In [3]:
import torch
import torch.optim as optim
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import random
import math
import pickle
import matplotlib.pyplot as plt
from torch import distributions as pyd
from torch.distributions import Categorical
from torch.optim.lr_scheduler import LambdaLR
import collections
from typing import Any, Dict, Optional
from tqdm.auto import tqdm

from utils import (
    rollout, relabel_action, generate_paths, get_expert_data,
    PolicyGaussian, PolicyAutoRegressiveModel, mlp
)
from policy import (
    torchify_dict, ConditionalUnet1D, Policy, BaseDiffusionDataset,
    create_sample_indices, get_data_stats, normalize_data, unnormalize_data
)

from evaluate import evaluate
import pytorch_utils as ptu
from reach_goal.envs.pointmaze_env import PointMazeEnv
from reach_goal.envs.pointmaze_expert import WaypointController

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

torch.manual_seed(0)
random.seed(0)
np.random.seed(0)

Using device: cuda:0


In [4]:
tmp = np.arange(10)
print(tmp)

[0 1 2 3 4 5 6 7 8 9]


In [5]:
np.random.shuffle(tmp)

In [6]:
print(tmp)

[2 8 4 9 1 6 7 3 0 5]


## 3. Behavior Cloning (BC)

Implement standard behavior cloning by maximizing log-likelihood of expert actions under the policy.

In [7]:
"""
TODO: MODIFY TO FILL IN YOUR BC IMPLEMENTATION
"""
def simulate_policy_bc(env, policy, expert_data, num_epochs=500, episode_length=50,
                       batch_size=32):

    # Hint: Use standard pytorch supervised learning code to train the policy.
    optimizer = optim.Adam(list(policy.parameters()), lr=1e-4)
    all_obs = np.concatenate([d['observations'] for d in expert_data])
    all_actions = np.concatenate([d['actions'] for d in expert_data])
    num_samples = all_obs.shape[0]
    # print: num_samples
    print("There are {} samples".format(num_samples)) # 250 samples
    num_batches = num_samples // batch_size # 250 // 32 = 7 num_batches
    losses = []
    # Convert to pytorch tensor:
    expert_obs = torch.tensor(all_obs, dtype=torch.float32)
    expert_action = torch.tensor(all_actions, dtype=torch.float32)

    for epoch in range(num_epochs):
        ## TODO Students
        order = np.arange(num_samples)
        np.random.shuffle(order) # suffle the order using in-place operation
        running_loss = 0.0

        # Apply shuffle to both observations and actions
        expert_obs = expert_obs[order]
        expert_action = expert_action[order]

        for i in range(num_batches): # for each iteration batch
            optimizer.zero_grad() # clear previous gradient first
            #========== TODO: start ==========
            train_obs = expert_obs[i*batch_size : (i+1)*batch_size].to(device)
            train_action = expert_action[i*batch_size : (i+1)*batch_size].to(device)

            # Sanity check
            if train_obs.size() == 0:
                continue


            # Sample action from the policy distribution and get log probability
            log_prob_action = policy.log_prob(train_obs,train_action)


            #  Compute loss: maximum log likelihood
            loss = -log_prob_action.mean()


            #========== TODO: end ==========
            loss.backward() # compute gradient of loss wrt to parameters
            optimizer.step() # update parameters
            running_loss += loss.item()
        # if epoch % 10 == 0:
        print('[%d] loss: %.8f' %
            (epoch, running_loss / num_batches))
        losses.append(loss.item())

## 4. DAgger (Dataset Aggregation)

Implement DAgger: iteratively train a policy, roll it out, relabel the collected trajectories using the expert, and add them to the training set.

In [8]:
"""
TODO: MODIFY TO FILL IN YOUR DAGGER IMPLEMENTATION
"""
def simulate_policy_dagger(env, policy, expert_paths, expert_policy=None, num_epochs=500, episode_length=50,
                            batch_size=32, num_dagger_iters=10, num_trajs_per_dagger=10):

    # Fill in your dagger implementation here.
    # Hint: Loop through num_dagger_iters iterations, at each iteration train a policy on the current dataset.
    # Then rollout the policy, use relabel_action to relabel the actions along the trajectory
    # with "expert_policy" and then add this to current dataset.
    # Repeat this so the dataset grows with states drawn from the policy, and relabeled actions using the expert.

    # Optimizer code
    optimizer = optim.Adam(list(policy.parameters()))
    losses = []
    returns = []

    trajs = expert_paths
    # Dagger iterations
    for dagger_itr in range(num_dagger_iters):
        all_obs = np.concatenate([d['observations'] for d in trajs])
        all_actions = np.concatenate([d['actions'] for d in trajs])
        num_samples = all_obs.shape[0]
        num_batches = num_samples // batch_size
        losses = []

        # Convert to pytorch tensor:
        expert_obs = torch.tensor(all_obs, dtype=torch.float32)
        expert_action = torch.tensor(all_actions, dtype=torch.float32)

        # Train the model with Adam
        for epoch in range(num_epochs):
            order = np.arange(num_samples)
            np.random.shuffle(order)
            running_loss = 0.0

            # Apply shuffle to both observations and actions
            expert_obs = expert_obs[order]
            expert_action = expert_action[order]

            for i in range(num_batches):
                optimizer.zero_grad()
                #========== TODO: start ==========
                train_obs = expert_obs[i*batch_size : (i+1)*batch_size].to(device)
                train_action = expert_action[i*batch_size : (i+1)*batch_size].to(device)

                # Sanity check
                if train_obs.size() == 0:
                    continue


                # Sample action from the policy distribution and get log probability
                log_prob_action = policy.log_prob(train_obs,train_action)


                #  Compute loss: maximum log likelihood
                loss = -log_prob_action.mean()




                #========== TODO: end ==========
                loss.backward()
                optimizer.step()

                # print statistics
                running_loss += loss.item()
            # if epoch % 10 == 0:
            print('[%d, %5d] loss: %.8f' %(epoch + 1, i + 1, running_loss/num_batches))
            losses.append(running_loss/num_batches)

        # Collecting more data for dagger
        trajs_recent = []
        for k in range(num_trajs_per_dagger):
            env.reset()
            #========== TODO: start ==========



            #========== TODO: end ==========

        trajs += trajs_recent
        mean_return = np.mean(np.array([traj['rewards'].sum() for traj in trajs_recent]))
        print("Average DAgger return is " + str(mean_return))
        returns.append(mean_return)

---
## 5. Training and Evaluation

Below we define helper functions and then run all the required training configurations.

In [ ]:
class Args:
    def __init__(self, env, train, policy, test=False, render=False):
        self.env = env       # 'reacher' or 'pointmaze'
        self.train = train   # 'behavior_cloning' or 'dagger' or 'diffusion'
        self.policy = policy # 'gaussian' or 'autoregressive' or 'diffusion'
        self.test = test
        self.render = render


def setup_and_run(args):
    """Load data, create env/policy, train, evaluate, and save weights."""
    # Get expert data
    if args.env == 'reacher':
        file_path = 'data/reacher_expert_data.pkl'
    elif args.env == 'pointmaze':
        file_path = 'data/pointmaze_expert_data.pkl'
    else:
        raise ValueError('Invalid environment')
    expert_data = get_expert_data(file_path)

    flattened_expert = {'observations': [], 'actions': []}
    for expert_path in expert_data:
        for k in flattened_expert.keys():
            flattened_expert[k].append(expert_path[k])
    for k in flattened_expert.keys():
        flattened_expert[k] = np.concatenate(flattened_expert[k])

    # Define environment
    if args.env == 'reacher':
        env = gym.make("Reacher-v4", render_mode='human' if args.render else None)
    elif args.env == 'pointmaze':
        env = PointMazeEnv(render_mode='human' if args.render else 'rgb_array')

    # Define policy
    hidden_dim = 128
    hidden_depth = 2
    obs_size = env.observation_space.shape[0]
    ac_size = env.action_space.shape[0]
    ac_margin = 0.1

    if args.policy == 'gaussian':
        policy = PolicyGaussian(num_inputs=obs_size, num_outputs=ac_size, hidden_dim=hidden_dim, hidden_depth=hidden_depth)
        policy.to(device)
    elif args.policy == 'autoregressive':
        num_buckets = 10
        policy = PolicyAutoRegressiveModel(num_inputs=obs_size, num_outputs=ac_size, hidden_dim=hidden_dim,
                                            hidden_depth=hidden_depth, num_buckets=num_buckets,
                                            ac_low=flattened_expert['actions'].min(axis=0) - ac_margin,
                                            ac_high=flattened_expert['actions'].max(axis=0) + ac_margin)
        policy.to(device)
    elif args.policy == 'diffusion':
        policy = DiffusionPolicy(obs_size=obs_size, obs_horizon=4, action_size=ac_size,
                                 action_horizon=8, action_pred_horizon=12,
                                 num_diffusion_iters=100, device=device)

    # Training hyperparameters
    if args.env == 'reacher':
        episode_length = 50
        num_epochs = 500
        batch_size = 32
    elif args.env == 'pointmaze':
        episode_length = 300
        num_epochs = 10
        batch_size = 128
        if args.train == 'diffusion':
            num_epochs = 1

    if not args.test:
        if args.train == 'behavior_cloning':
            simulate_policy_bc(env, policy, expert_data, num_epochs=num_epochs,
                               episode_length=episode_length, batch_size=batch_size)
        elif args.train == 'dagger':
            if args.env == 'reacher':
                expert_policy = torch.load('data/reacher_expert_policy.pkl', map_location=torch.device(device), weights_only=False)
                print("Expert policy loaded")
                expert_policy.to(device)
                ptu.set_gpu_mode(torch.cuda.is_available())
            elif args.env == 'pointmaze':
                expert_policy = WaypointController(env.maze)

            num_dagger_iters = 10
            num_epochs = int(num_epochs / num_dagger_iters)
            num_trajs_per_dagger = 10

            simulate_policy_dagger(env, policy, expert_data, expert_policy,
                                   num_epochs=num_epochs, episode_length=episode_length,
                                   batch_size=batch_size, num_dagger_iters=num_dagger_iters,
                                   num_trajs_per_dagger=num_trajs_per_dagger)
        elif args.train == 'diffusion' and args.env == 'pointmaze':
            policy = train_diffusion_policy(policy, get_expert_data(file_path),
                                           num_epochs=num_epochs, batch_size=batch_size)
        else:
            raise ValueError('Invalid training method')

        torch.save(policy.state_dict(), f'{args.policy}_{args.env}_{args.train}_final.pth')
    else:
        policy.load_state_dict(torch.load(f'{args.policy}_{args.env}_{args.train}_final.pth', weights_only=True))

    evaluate(env, policy, args.train, num_validation_runs=100, episode_length=episode_length,
             render=args.render, env_name=args.env)
    env.close()
    return policy

### 6a. Gaussian BC — Reacher

In [ ]:
args = Args('reacher', 'behavior_cloning', 'gaussian')
policy_gaussian_reacher_bc = setup_and_run(args)

### 6b. Gaussian DAgger — Reacher

In [ ]:
args = Args('reacher', 'dagger', 'gaussian')
policy_gaussian_reacher_dagger = setup_and_run(args)

### 6c. Gaussian BC — PointMaze

In [ ]:
args = Args('pointmaze', 'behavior_cloning', 'gaussian')
policy_gaussian_pointmaze_bc = setup_and_run(args)

### 6d. Gaussian DAgger — PointMaze

In [ ]:
args = Args('pointmaze', 'dagger', 'gaussian')
policy_gaussian_pointmaze_dagger = setup_and_run(args)